# Practical 3: A first neural network

**Intro to AI, Machine Learning and Data Science** · about 25 minutes

**Before you start:** submit the batch job for this practical so that it runs while you work through the notebook. In a terminal on MeluXina, inside this folder:

```bash
sbatch train_digits.slurm
```

We will look at its output at the end.

In this notebook you will:

1. Look at a small **image** dataset: 8 by 8 pixel handwritten digits
2. Train a **multi-layer perceptron** (MLP) and read its **training curve**
3. Compare it with the classical models from Practical 2
4. Open the box: reproduce the network's predictions with **plain NumPy matrix multiplications**
5. Collect the results of your **batch job**

*Data:* UCI ML hand-written digits (Alpaydin & Kaynak, 1998), bundled with scikit-learn.

In [ ]:
%matplotlib inline
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "lines.linewidth": 2,
    "axes.prop_cycle": plt.cycler(color=["#2a78d6", "#eb6834", "#1baf7a", "#eda100",
                                         "#e87ba4", "#008300", "#4a3aa7", "#e34948"]),
})

## 1. The data: images as feature vectors

Each image is an 8 by 8 grid of grey levels from 0 to 16. For a simple network we **flatten** it into a vector of 64 features.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X, y = digits.data / 16.0, digits.target     # scale pixel values to [0, 1]
print("X shape:", X.shape, " classes:", np.unique(y))

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for ax, image, label in zip(axes, digits.images, digits.target):
    ax.imshow(image, cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=0
)
print(f"train: {X_train.shape}, test: {X_test.shape}")

## 2. Train a multi-layer perceptron

`MLPClassifier(hidden_layer_sizes=(64,))` builds a network with **64 inputs → 64 hidden ReLU neurons → 10 outputs**, trained with the Adam optimiser on mini-batches. It is the same recipe as Practical 1: model, loss (cross-entropy), gradients (by backpropagation), and gradient steps. One **epoch** is one full pass over the training data.

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=0)
start = time.perf_counter()
mlp.fit(X_train, y_train)
print(f"trained for {mlp.n_iter_} epochs in {time.perf_counter() - start:.1f} s")
print(f"train accuracy: {mlp.score(X_train, y_train):.3f}")
print(f"test accuracy : {mlp.score(X_test, y_test):.3f}")

fig, ax = plt.subplots()
ax.plot(mlp.loss_curve_)
ax.set(xlabel="epoch", ylabel="training loss (cross-entropy)", yscale="log",
       title="The training curve: loss goes down as the network learns");

How many parameters did the network learn? Each layer has a weight matrix and a bias vector.

In [ ]:
for i, (W, b) in enumerate(zip(mlp.coefs_, mlp.intercepts_)):
    print(f"layer {i + 1}: weights {W.shape}, biases {b.shape}")
n_params = sum(W.size for W in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)
print(f"total parameters: {n_params:,}")

A training accuracy of 1.000 with a lower test accuracy is typical: the network has memorised its training set. The test number is the one to report.

## 3. Is the neural network better than classical ML?

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

rivals = {
    "logistic regression": LogisticRegression(max_iter=2000),
    "random forest": RandomForestClassifier(random_state=0),
    "SVM (RBF kernel)": SVC(),
    "MLP (64 hidden)": MLPClassifier(hidden_layer_sizes=(64,), max_iter=1000, random_state=0),
}
rows = []
for name, model in rivals.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    rows.append({"model": name, "test accuracy": model.score(X_test, y_test),
                 "fit time (s)": time.perf_counter() - start})
pd.DataFrame(rows).set_index("model").round(3)

On a small dataset of 1,347 tiny images, the neural network is **good but not magic**: a well-chosen classical model does as well or better, and trains faster. Neural networks pull ahead when data is large, high-dimensional and structured (full-size images, sequences, molecules, simulations), and when the architecture matches that structure (convolutions for images, graphs for molecules). That is where GPUs and HPC come in.

### Your turn

Try a few network sizes, e.g. `(16,)`, `(64,)`, `(256,)` and `(128, 64)` (two hidden layers). For each, record train accuracy, test accuracy, number of parameters and fit time. What do you observe?

In [ ]:
# Your code here

<details>
<summary>Show solution</summary>

```python
rows = []
for sizes in [(16,), (64,), (256,), (128, 64)]:
    model = MLPClassifier(hidden_layer_sizes=sizes, max_iter=1000, random_state=0)
    start = time.perf_counter()
    model.fit(X_train, y_train)
    rows.append({
        "hidden layers": str(sizes),
        "parameters": sum(W.size for W in model.coefs_) + sum(b.size for b in model.intercepts_),
        "train acc": model.score(X_train, y_train),
        "test acc": model.score(X_test, y_test),
        "fit time (s)": time.perf_counter() - start,
    })
pd.DataFrame(rows).round(3)
```

The number of parameters grows sixteenfold from the smallest to the largest network, while test accuracy only moves between about 0.97 and 0.99. More capacity only helps if the data can support it. Regularisation (the `alpha` parameter, an L2 penalty on the weights) and early stopping (`early_stopping=True`, which holds back part of the training data as a validation set) are the usual ways to control overfitting.

</details>

## 4. Open the box: a neural network is matrix multiplications

For the hidden layer, each neuron computes a weighted sum of its inputs plus a bias and applies the ReLU activation $\max(0, z)$. For a whole batch of images at once, that is a **matrix multiplication**:

$$H = \mathrm{ReLU}(X W_1 + b_1), \qquad \text{scores} = H W_2 + b_2, \qquad p = \mathrm{softmax}(\text{scores})$$

Let's recompute the network's predictions with plain NumPy using the learned weights.

In [ ]:
W1, W2 = mlp.coefs_
b1, b2 = mlp.intercepts_

H = np.maximum(0, X_test @ W1 + b1)          # hidden layer: matrix multiply + ReLU
scores = H @ W2 + b2                         # output layer: another matrix multiply
p = np.exp(scores - scores.max(axis=1, keepdims=True))
p /= p.sum(axis=1, keepdims=True)            # softmax: scores -> probabilities

print("matches scikit-learn:", np.allclose(p, mlp.predict_proba(X_test)))
print("first 10 predictions:", p.argmax(axis=1)[:10])
print("true labels         :", y_test[:10])

That's all a trained network does at prediction time: multiply matrices and apply simple non-linear functions. Training adds the backward pass, which is more matrix multiplications. **GPUs are extremely fast at matrix multiplication**, which is why deep learning runs on them, for example on MeluXina's GPU nodes.

Finally, which digits does the network get wrong?

In [ ]:
pred = mlp.predict(X_test)
wrong = np.flatnonzero(pred != y_test)
print(f"{len(wrong)} of {len(y_test)} test images misclassified")

fig, axes = plt.subplots(1, min(8, len(wrong)), figsize=(10, 1.8))
for ax, i in zip(np.atleast_1d(axes), wrong[:8]):
    ax.imshow(X_test[i].reshape(8, 8), cmap="gray_r")
    ax.set_title(f"true {y_test[i]}\npred {pred[i]}", fontsize=10)
    ax.axis("off")

Many mistakes are digits a human would also hesitate over at 8 by 8 pixels. Looking at failures is one of the most useful habits in ML: it tells you whether you need more data, better features, or a better model.

## 5. Your batch job

The script `train_digits.py` runs a **hyperparameter search** for this MLP (layer sizes × regularisation × learning rates, each checked with 5-fold cross-validation). It runs the search twice, first on 1 CPU core and then on all the cores Slurm gave the job. Before the parallel search it starts the worker processes and times that separately, because each worker first has to import NumPy, SciPy and scikit-learn.

Check on it in a terminal:

```bash
squeue --me                 # is it still running?
cat digits-<jobid>.out      # the results
```

Questions to discuss with your neighbour:

* What speed-up did the search get on 16 cores? Why is it less than 16?
* How long did starting the workers take, compared with the search itself? When is parallelism worth it?
* Is the best configuration much better than the default one we used above?
* How would you run a search that is 100 times bigger? *Hint:* Slurm job arrays, or many nodes.

## Summary

* A neural network is layers of **weighted sums + non-linear activations**, trained with the same recipe as any ML model: loss, gradients (backpropagation), gradient steps.
* Watch the **training curve**; report **test** performance.
* On small tabular-like data, classical ML is often just as good. Deep learning shines on large, structured data.
* Under the hood it is **matrix multiplication**, which is why **GPUs** matter.
* Hyperparameter searches are **embarrassingly parallel**: a natural fit for an HPC system.

**Next steps:** larger networks and datasets, trained on GPUs.